# Beauty Text Embedding Generation

This notebook generates Beauty-only item text embeddings for CBPR-style recommendation.

It follows the mature procedure in `Beauty_text_embedding_generating.md`:

1. Reproduce Yun Chuan's Beauty review filtering logic.
2. Derive the final Beauty `parent_asin` item universe from train + valid + test.
3. Filter metadata to that item universe only.
4. Build 4-column product text from `title`, `categories`, `features`, and `description`.
5. Encode text with `BAAI/bge-base-en-v1.5` as a frozen feature extractor.
6. Store embeddings as resumable float32 Parquet chunks.
7. Produce manifests, QA files, and a consolidated `summary.json`.
8. Provide tensor helpers for final-item-index debug tensors and RecBole-aligned CBPR tensors.

Default note: this notebook derives the full Yun Chuan-filtered Beauty item universe, then embeds only the first 500 items because `MAX_FINAL_ITEMS = 500`. Set `MAX_FINAL_ITEMS = None` for the full production embedding run.

The final RecBole-aligned `.pt` tensor can only be built safely when the RecBole `dataset` object is available, because RecBole may remap item tokens internally. If a `dataset` variable exists, the final run cell will build the aligned tensor automatically; otherwise it writes the embedding cache and clearly reports that tensor alignment is pending.

In [ ]:
from __future__ import annotations

import csv
import gc
import html
import json
import os
import re
import sqlite3
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


PROJECT_DIR = Path('/Users/frankwang1224/Projects/rcd_sys_proj02')
DATASET_DIR = PROJECT_DIR / 'dataset'
REVIEW_PATH = DATASET_DIR / 'Beauty_and_Personal_Care.jsonl'
META_PATH = DATASET_DIR / 'meta_Beauty_and_Personal_Care.jsonl'

OUTPUT_DIR = PROJECT_DIR / 'embeddings' / 'text_bge_base_en_v1_5_4col_beauty_yc_filtered'
CHUNKS_DIR = OUTPUT_DIR / 'chunks'
SUMMARY_DIR = OUTPUT_DIR / '_summary'
WORK_DIR = OUTPUT_DIR / '_working'

ITEM_UNIVERSE_PATH = OUTPUT_DIR / 'beauty_final_item_universe.csv'
PREPROCESSING_MANIFEST_PATH = OUTPUT_DIR / 'text_preprocessing_manifest.csv'
TEXT_MANIFEST_PATH = OUTPUT_DIR / 'text_manifest.csv'
COMBINED_PARQUET_PATH = OUTPUT_DIR / 'text_embeddings_by_parent_asin.parquet'
FINAL_INDEX_DEBUG_TENSOR_PATH = OUTPUT_DIR / 'beauty_bge_base_en_v1_5_text_embeddings_by_final_item_idx_debug.pt'
RECBOLE_ALIGNED_TENSOR_PATH = OUTPUT_DIR / 'beauty_bge_base_en_v1_5_text_embeddings_recbole_aligned.pt'
SUMMARY_JSON_PATH = OUTPUT_DIR / 'summary.json'
SQLITE_PATH = WORK_DIR / 'beauty_review_filtering.sqlite'

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
MODEL_MAX_SEQ_LENGTH = 512
EMBEDDING_DIM = 768
EMBEDDING_DTYPE = 'float32'
PREPROCESSING_VERSION = 'beauty_text4col_yc_filtered_v2_2026_06_23'

START_DATE = '2021-01-01'
END_DATE = '2022-12-31'
TRAIN_END_CUTOFF_DATE = '2022-08-01'
VALID_END_CUTOFF_DATE = '2022-10-01'
USER_MIN_REVIEWS = 5
WARM_USER_MIN_REVIEWS = 10
WARM_ITEM_MIN_REVIEWS = 5

BATCH_SIZE = 64
CHUNK_SIZE = 10_000
DEVICE: str | None = None
MODEL_LOCAL_FILES_ONLY = True

# Embedding validation default. Set MAX_FINAL_ITEMS to None for the full production embedding run.
# Keep MAX_REVIEW_ROWS as None for the correct full Yun Chuan-filtered item universe.
MAX_REVIEW_ROWS: int | None = None
MAX_FINAL_ITEMS: int | None = 500
RUN_MODE = 'validation' if MAX_FINAL_ITEMS is not None else 'production'

# Set True only when you intentionally want to rebuild outputs from raw reviews.
FORCE_REBUILD_ITEM_UNIVERSE = False

# Keep vector Parquet compact. Full source text is saved only in audit samples.
SAVE_EMBEDDING_TEXT_IN_PARQUET = False
TEXT_SAMPLE_LIMIT = 50
TOKEN_AWARE_TRUNCATION = True
WRITE_COMBINED_PARQUET = True
BUILD_FINAL_INDEX_DEBUG_TENSOR = False
AUTO_BUILD_RECBOLE_TENSOR_IF_DATASET_EXISTS = True

for path in [OUTPUT_DIR, CHUNKS_DIR, SUMMARY_DIR, WORK_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Project dir: {PROJECT_DIR}')
print(f'Review path: {REVIEW_PATH}')
print(f'Metadata path: {META_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Run mode: {RUN_MODE}')
print(f'Max review rows: {MAX_REVIEW_ROWS}')
print(f'Max final items: {MAX_FINAL_ITEMS}')


## Stage A: Reproduce Yun Chuan's Beauty Filtering Logic

This stage streams the raw Beauty review JSONL and stores the latest review for each `(user_id, parent_asin)` pair in SQLite. SQLite keeps this memory-safe for the large review file.

The logic is:

- keep reviews from `2021-01-01` through `2022-12-31`;
- deduplicate `(user_id, parent_asin)`, keeping the latest timestamp;
- remove users with fewer than 5 Beauty interactions;
- split by timestamp into train/valid/test;
- keep valid/test users only if they appear in train;
- define the final text-embedding item universe from `train + valid + test`.

Warm/cold labels are calculated after the train split and are not used to drop users.

The item universe file uses a dense `final_item_idx` over the final item universe only. It intentionally avoids the misleading sparse `yc_iid` debug index from the earlier draft.

In [ ]:
def date_to_ms(date_str: str) -> int:
    dt = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


def sqlite_table_exists(connection: sqlite3.Connection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (table_name,),
    ).fetchone()
    return row is not None


def sqlite_row_count(connection: sqlite3.Connection, table_name: str) -> int:
    if not sqlite_table_exists(connection, table_name):
        return 0
    return int(connection.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0])


def iter_review_jsonl(path: Path, max_rows: int | None = None) -> Iterable[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if max_rows is not None and line_number > max_rows:
                break
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue


def initialize_review_db(connection: sqlite3.Connection) -> None:
    connection.execute('PRAGMA journal_mode=WAL')
    connection.execute('PRAGMA synchronous=NORMAL')
    connection.execute('PRAGMA temp_store=MEMORY')
    connection.execute(
        '''
        CREATE TABLE IF NOT EXISTS latest_reviews (
            user_id TEXT NOT NULL,
            parent_asin TEXT NOT NULL,
            rating REAL,
            timestamp INTEGER NOT NULL,
            PRIMARY KEY (user_id, parent_asin)
        )
        '''
    )
    connection.commit()


def ingest_latest_reviews(connection: sqlite3.Connection) -> dict[str, int]:
    start_ts = date_to_ms(START_DATE)
    end_ts = date_to_ms(END_DATE)
    insert_sql = '''
        INSERT INTO latest_reviews (user_id, parent_asin, rating, timestamp)
        VALUES (?, ?, ?, ?)
        ON CONFLICT(user_id, parent_asin) DO UPDATE SET
            rating = excluded.rating,
            timestamp = excluded.timestamp
        WHERE excluded.timestamp > latest_reviews.timestamp
    '''

    scanned_rows = 0
    kept_rows = 0
    skipped_rows = 0
    batch: list[tuple[str, str, float | None, int]] = []
    batch_size = 50_000

    progress = tqdm(iter_review_jsonl(REVIEW_PATH, MAX_REVIEW_ROWS), desc='stream reviews', unit='row')
    for row in progress:
        scanned_rows += 1
        user_id = row.get('user_id')
        parent_asin = row.get('parent_asin')
        timestamp = row.get('timestamp')
        rating = row.get('rating')

        if user_id is None or parent_asin is None or timestamp is None:
            skipped_rows += 1
            continue

        try:
            ts_int = int(timestamp)
        except (TypeError, ValueError):
            skipped_rows += 1
            continue

        if ts_int < start_ts or ts_int > end_ts:
            continue

        rating_value = float(rating) if rating is not None else None
        batch.append((str(user_id), str(parent_asin), rating_value, ts_int))
        kept_rows += 1

        if len(batch) >= batch_size:
            connection.executemany(insert_sql, batch)
            connection.commit()
            batch = []
            progress.set_postfix(kept=f'{kept_rows:,}')

    if batch:
        connection.executemany(insert_sql, batch)
        connection.commit()

    connection.execute('CREATE INDEX IF NOT EXISTS idx_latest_user ON latest_reviews(user_id)')
    connection.execute('CREATE INDEX IF NOT EXISTS idx_latest_parent ON latest_reviews(parent_asin)')
    connection.execute('CREATE INDEX IF NOT EXISTS idx_latest_ts ON latest_reviews(timestamp)')
    connection.commit()

    return {
        'review_rows_scanned': scanned_rows,
        'review_rows_in_date_window_before_dedupe': kept_rows,
        'review_rows_skipped_bad_keys_or_ts': skipped_rows,
        'latest_review_pairs_after_dedupe': sqlite_row_count(connection, 'latest_reviews'),
    }


def derive_beauty_splits(connection: sqlite3.Connection) -> dict[str, Any]:
    train_end_ts = date_to_ms(TRAIN_END_CUTOFF_DATE)
    valid_end_ts = date_to_ms(VALID_END_CUTOFF_DATE)

    for table_name in [
        'valid_users',
        'valid_reviews',
        'train_reviews',
        'train_users',
        'valid_split_reviews',
        'test_split_reviews',
        'final_items',
        'final_item_map',
        'user_train_counts',
        'item_train_counts',
    ]:
        connection.execute(f'DROP TABLE IF EXISTS {table_name}')

    connection.execute(
        '''
        CREATE TABLE valid_users AS
        SELECT user_id, COUNT(*) AS n_interactions
        FROM latest_reviews
        GROUP BY user_id
        HAVING COUNT(*) >= ?
        ''',
        (USER_MIN_REVIEWS,),
    )
    connection.execute('CREATE INDEX idx_valid_users_user ON valid_users(user_id)')

    connection.execute(
        '''
        CREATE TABLE valid_reviews AS
        SELECT r.*
        FROM latest_reviews r
        JOIN valid_users u ON r.user_id = u.user_id
        '''
    )
    connection.execute('CREATE INDEX idx_valid_reviews_user ON valid_reviews(user_id)')
    connection.execute('CREATE INDEX idx_valid_reviews_parent ON valid_reviews(parent_asin)')
    connection.execute('CREATE INDEX idx_valid_reviews_ts ON valid_reviews(timestamp)')

    connection.execute(
        'CREATE TABLE train_reviews AS SELECT * FROM valid_reviews WHERE timestamp <= ?',
        (train_end_ts,),
    )
    connection.execute('CREATE INDEX idx_train_reviews_user ON train_reviews(user_id)')
    connection.execute('CREATE INDEX idx_train_reviews_parent ON train_reviews(parent_asin)')

    connection.execute('CREATE TABLE train_users AS SELECT DISTINCT user_id FROM train_reviews')
    connection.execute('CREATE INDEX idx_train_users_user ON train_users(user_id)')

    connection.execute(
        '''
        CREATE TABLE valid_split_reviews AS
        SELECT r.*
        FROM valid_reviews r
        JOIN train_users t ON r.user_id = t.user_id
        WHERE r.timestamp > ? AND r.timestamp <= ?
        ''',
        (train_end_ts, valid_end_ts),
    )

    connection.execute(
        '''
        CREATE TABLE test_split_reviews AS
        SELECT r.*
        FROM valid_reviews r
        JOIN train_users t ON r.user_id = t.user_id
        WHERE r.timestamp > ?
        ''',
        (valid_end_ts,),
    )

    connection.execute(
        '''
        CREATE TABLE final_items AS
        SELECT DISTINCT parent_asin FROM train_reviews
        UNION
        SELECT DISTINCT parent_asin FROM valid_split_reviews
        UNION
        SELECT DISTINCT parent_asin FROM test_split_reviews
        '''
    )
    connection.execute('CREATE INDEX idx_final_items_parent ON final_items(parent_asin)')

    # Dense index over the final embedding item universe only. This is a debug/alignment helper,
    # not a substitute for RecBole's internal item id mapping.
    connection.execute(
        '''
        CREATE TABLE final_item_map AS
        SELECT parent_asin, ROW_NUMBER() OVER (ORDER BY parent_asin) AS final_item_idx
        FROM final_items
        '''
    )
    connection.execute('CREATE INDEX idx_final_item_map_parent ON final_item_map(parent_asin)')

    connection.execute(
        '''
        CREATE TABLE user_train_counts AS
        SELECT user_id, COUNT(*) AS num_train
        FROM train_reviews
        GROUP BY user_id
        '''
    )
    connection.execute(
        '''
        CREATE TABLE item_train_counts AS
        SELECT parent_asin, COUNT(*) AS num_train
        FROM train_reviews
        GROUP BY parent_asin
        '''
    )
    connection.commit()

    split_rows: list[dict[str, Any]] = []
    for split_name, table_name in [('train', 'train_reviews'), ('valid', 'valid_split_reviews'), ('test', 'test_split_reviews')]:
        row = connection.execute(
            f'''
            SELECT
                COUNT(*) AS interactions,
                COUNT(DISTINCT user_id) AS users,
                COUNT(DISTINCT parent_asin) AS items
            FROM {table_name}
            '''
        ).fetchone()
        split_rows.append({'split': split_name, 'interactions': int(row[0]), 'users': int(row[1]), 'items': int(row[2])})

    split_summary_path = SUMMARY_DIR / 'beauty_review_split_summary.csv'
    pd.DataFrame(split_rows).to_csv(split_summary_path, index=False)

    warm_users = int(connection.execute('SELECT COUNT(*) FROM user_train_counts WHERE num_train >= ?', (WARM_USER_MIN_REVIEWS,)).fetchone()[0])
    cold_users = int(connection.execute('SELECT COUNT(*) FROM user_train_counts WHERE num_train < ?', (WARM_USER_MIN_REVIEWS,)).fetchone()[0])
    warm_items = int(connection.execute('SELECT COUNT(*) FROM item_train_counts WHERE num_train >= ?', (WARM_ITEM_MIN_REVIEWS,)).fetchone()[0])
    cold_items = int(connection.execute('SELECT COUNT(*) FROM item_train_counts WHERE num_train < ?', (WARM_ITEM_MIN_REVIEWS,)).fetchone()[0])

    item_universe_df = pd.read_sql_query(
        '''
        SELECT parent_asin, final_item_idx
        FROM final_item_map
        ORDER BY final_item_idx
        ''',
        connection,
    )
    item_universe_df.to_csv(ITEM_UNIVERSE_PATH, index=False)

    return {
        'latest_review_pairs_after_dedupe': sqlite_row_count(connection, 'latest_reviews'),
        'users_after_user_min_filter': sqlite_row_count(connection, 'valid_users'),
        'reviews_after_user_min_filter': sqlite_row_count(connection, 'valid_reviews'),
        'train_interactions': split_rows[0]['interactions'],
        'valid_interactions': split_rows[1]['interactions'],
        'test_interactions': split_rows[2]['interactions'],
        'final_items': int(len(item_universe_df)),
        'final_item_idx_min': int(item_universe_df['final_item_idx'].min()) if len(item_universe_df) else None,
        'final_item_idx_max': int(item_universe_df['final_item_idx'].max()) if len(item_universe_df) else None,
        'warm_users_by_train_history': warm_users,
        'cold_users_by_train_history': cold_users,
        'warm_items_by_train_history': warm_items,
        'cold_items_by_train_history': cold_items,
        'split_summary_path': str(split_summary_path),
        'item_universe_path': str(ITEM_UNIVERSE_PATH),
    }


def existing_item_universe_is_current(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        columns = set(pd.read_csv(path, nrows=1).columns)
    except Exception:
        return False
    return {'parent_asin', 'final_item_idx'}.issubset(columns)


def ensure_item_universe() -> pd.DataFrame:
    if existing_item_universe_is_current(ITEM_UNIVERSE_PATH) and not FORCE_REBUILD_ITEM_UNIVERSE:
        print(f'Using existing item universe: {ITEM_UNIVERSE_PATH}')
        return pd.read_csv(ITEM_UNIVERSE_PATH)

    if ITEM_UNIVERSE_PATH.exists() and not existing_item_universe_is_current(ITEM_UNIVERSE_PATH):
        print('Existing item universe file uses an old schema; rebuilding with dense final_item_idx.')

    if FORCE_REBUILD_ITEM_UNIVERSE and SQLITE_PATH.exists():
        SQLITE_PATH.unlink()

    with sqlite3.connect(SQLITE_PATH) as connection:
        initialize_review_db(connection)
        if sqlite_row_count(connection, 'latest_reviews') == 0:
            ingest_summary = ingest_latest_reviews(connection)
        else:
            ingest_summary = {
                'review_rows_scanned': None,
                'review_rows_in_date_window_before_dedupe': None,
                'review_rows_skipped_bad_keys_or_ts': None,
                'latest_review_pairs_after_dedupe': sqlite_row_count(connection, 'latest_reviews'),
                'reused_existing_latest_reviews_table': True,
            }
        split_summary = derive_beauty_splits(connection)

    summary = {
        'stage': 'item_universe',
        'review_path': str(REVIEW_PATH),
        'start_date': START_DATE,
        'end_date': END_DATE,
        'train_end_cutoff_date': TRAIN_END_CUTOFF_DATE,
        'valid_end_cutoff_date': VALID_END_CUTOFF_DATE,
        'user_min_reviews': USER_MIN_REVIEWS,
        'warm_user_min_reviews': WARM_USER_MIN_REVIEWS,
        'warm_item_min_reviews': WARM_ITEM_MIN_REVIEWS,
        'max_review_rows': MAX_REVIEW_ROWS,
        **ingest_summary,
        **split_summary,
        'created_at_unix': time.time(),
    }
    item_universe_summary_path = SUMMARY_DIR / 'beauty_item_universe_summary.json'
    item_universe_summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2))
    return pd.read_csv(ITEM_UNIVERSE_PATH)


item_universe_df = ensure_item_universe()
print(item_universe_df.head())
print(f'Final Beauty item universe rows: {len(item_universe_df):,}')


## Stage B: Four-Column Text Preprocessing

The embedding text is built as:

```text
Title: ... Categories: ... Features: ... Description: ...
```

Cleaning is light and product-preserving. Seller-service boilerplate is removed only from `features` and `description`.

The service-noise patterns are intentionally narrower than the first draft. For example, the notebook removes phrases like `free shipping`, `return policy`, and `amazon prime`, but it does not remove product-copy sentences merely because they contain words like `prime`, `delivery`, or `guarantee`.

In [ ]:
def choose_device() -> str:
    if DEVICE is not None:
        return DEVICE
    if torch.cuda.is_available():
        return 'cuda'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


selected_device = choose_device()
print(f'Loading {MODEL_NAME} on {selected_device}')
if MODEL_LOCAL_FILES_ONLY:
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

try:
    model = SentenceTransformer(MODEL_NAME, device=selected_device, local_files_only=MODEL_LOCAL_FILES_ONLY)
except TypeError:
    # Older SentenceTransformers versions may not expose local_files_only directly;
    # the offline environment variables above still prevent network access.
    model = SentenceTransformer(MODEL_NAME, device=selected_device)
except Exception as exc:
    if MODEL_LOCAL_FILES_ONLY:
        raise RuntimeError(
            f'Could not load {MODEL_NAME} from the local Hugging Face cache. '
            'Set MODEL_LOCAL_FILES_ONLY = False when network access is available, '
            'or pre-download the model before running the full embedding job.'
        ) from exc
    raise
model.max_seq_length = MODEL_MAX_SEQ_LENGTH
TOKENIZER = model.tokenizer
print(model)
print('max_seq_length:', model.max_seq_length)


In [ ]:
TEXT_FIELDS = ['title', 'categories', 'features', 'description']

TAG_RE = re.compile(r'<[^>]+>')
SPACE_RE = re.compile(r'\s+')
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|[;\n\r]+')
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b')
DECORATIVE_OR_CONTROL_CATEGORIES = {'So', 'Sk', 'Cc', 'Cf'}

SERVICE_PATTERNS = {
    'refund': r'\b(?:full\s+)?refunds?\b|\brefund policy\b',
    'return': r'\breturn policy\b|\bfree returns?\b|\beasy returns?\b|\bhassle[- ]?free returns?\b|\b\d+[- ]?day returns?\b',
    'replacement': r'\bfree replacements?\b|\breplacement guarantee\b|\breplacements?\s+or\s+refunds?\b',
    'warranty': r'\b(?:\d+[- ]?)?(?:year|month|lifetime)\s+warrant(?:y|ies)\b|\bwarranty service\b',
    'guarantee': r'\bsatisfaction guaranteed\b|\bmoney[- ]?back guarantee\b|\brisk[- ]?free\b',
    'customer_service': r'\bcustomer service\b|\bafter[- ]?sales service\b|\bafter sales service\b',
    'contact': r'\bcontact us\b|\bcontact seller\b|\bfeel free to contact\b|\bemail us\b',
    'shipping_delivery': r'\bfree shipping\b|\bfast shipping\b|\bshipping policy\b|\bships? within\b|\bdelivery time\b|\bdelivered by\b',
    'amazon_fba_prime': r'\bamazon fba\b|\bfulfilled by amazon\b|\bamazon prime\b|\bprime shipping\b',
}
SERVICE_NOISE_RE = re.compile('|'.join(f'(?:{pattern})' for pattern in SERVICE_PATTERNS.values()), re.IGNORECASE)


@dataclass
class PreparedTextItem:
    parent_asin: str
    final_item_idx: int
    store: str | None
    price: str | None
    embedding_text: str
    has_title: bool
    has_categories: bool
    has_features: bool
    has_description: bool
    embedding_text_chars: int
    embedding_text_tokens: int | None
    text_was_token_truncated: bool
    text_fail_reason: str | None = None


def clean_scalar(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def value_to_raw_parts(value: Any) -> list[str]:
    if value is None:
        return []
    values = value if isinstance(value, list) else [value]
    parts: list[str] = []
    for item in values:
        if item is None:
            continue
        if isinstance(item, dict):
            text = json.dumps(item, ensure_ascii=False, sort_keys=True)
        else:
            text = str(item)
        text = text.strip()
        if text:
            parts.append(text)
    return parts


def normalize_unicode_for_embedding(text: str) -> str:
    normalized = unicodedata.normalize('NFKC', text)
    kept: list[str] = []
    for char in normalized:
        category = unicodedata.category(char)
        if category in DECORATIVE_OR_CONTROL_CATEGORIES:
            kept.append(' ')
        else:
            kept.append(char)
    return ''.join(kept)


def light_normalize_text(text: str) -> str:
    text = html.unescape(text)
    text = TAG_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = EMAIL_RE.sub(' ', text)
    text = normalize_unicode_for_embedding(text)
    text = SPACE_RE.sub(' ', text)
    return text.strip()


def split_sentence_like_pieces(text: str) -> list[str]:
    pieces = [piece.strip() for piece in SENTENCE_SPLIT_RE.split(text) if piece.strip()]
    return pieces if pieces else ([text.strip()] if text.strip() else [])


def join_clean_parts(field_name: str, parts: list[str]) -> str:
    if not parts:
        return ''
    if field_name == 'categories':
        return ' > '.join(parts)
    if field_name == 'features':
        return '; '.join(parts)
    if field_name == 'description':
        return ' '.join(parts)
    return ' '.join(parts)


def clean_field(field_name: str, value: Any) -> str:
    raw_parts = value_to_raw_parts(value)
    clean_parts: list[str] = []
    seen: set[str] = set()

    for raw_part in raw_parts:
        normalized_part = light_normalize_text(raw_part)
        if not normalized_part:
            continue

        candidate_pieces = split_sentence_like_pieces(normalized_part) if field_name in {'features', 'description'} else [normalized_part]
        for piece in candidate_pieces:
            if field_name in {'features', 'description'} and SERVICE_NOISE_RE.search(piece):
                continue
            key = piece.casefold()
            if key in seen:
                continue
            seen.add(key)
            clean_parts.append(piece)

    return join_clean_parts(field_name, clean_parts)


def split_clean_field_for_dedup(field_name: str, text: str) -> list[str]:
    if not text:
        return []
    if field_name == 'features':
        return [piece.strip() for piece in text.split(';') if piece.strip()]
    if field_name == 'description':
        return split_sentence_like_pieces(text)
    return [text.strip()]


def deduplicate_fields_across_item(field_texts: dict[str, str]) -> dict[str, str]:
    seen: set[str] = set()
    deduped: dict[str, str] = {}
    for field_name in TEXT_FIELDS:
        kept_parts: list[str] = []
        for piece in split_clean_field_for_dedup(field_name, field_texts.get(field_name, '')):
            key = piece.casefold()
            if key in seen:
                continue
            seen.add(key)
            kept_parts.append(piece)
        deduped[field_name] = join_clean_parts(field_name, kept_parts)
    return deduped


def build_labeled_text(field_texts: Mapping[str, str]) -> str:
    labels = {'title': 'Title', 'categories': 'Categories', 'features': 'Features', 'description': 'Description'}
    sections: list[str] = []
    for field_name in TEXT_FIELDS:
        value = field_texts.get(field_name, '').strip()
        if value:
            sections.append(f'{labels[field_name]}: {value}')
    return SPACE_RE.sub(' ', ' '.join(sections)).strip()


def count_bge_tokens(text: str) -> int:
    if not text:
        return 0
    return int(len(TOKENIZER(text, add_special_tokens=True, truncation=False, verbose=False)['input_ids']))


def truncate_on_word(text: str, max_chars: int) -> str:
    if max_chars <= 0:
        return ''
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(' ', 1)[0].strip()


def shrink_field_to_fit(field_texts: dict[str, str], field_name: str, max_tokens: int) -> dict[str, str]:
    original = field_texts.get(field_name, '')
    if not original:
        return field_texts

    low = 0
    high = len(original)
    best = ''
    while low <= high:
        mid = (low + high) // 2
        candidate = truncate_on_word(original, mid)
        trial = dict(field_texts)
        trial[field_name] = candidate
        token_count = count_bge_tokens(build_labeled_text(trial))
        if token_count <= max_tokens:
            best = candidate
            low = mid + 1
        else:
            high = mid - 1

    result = dict(field_texts)
    result[field_name] = best
    return result


def fit_text_to_budget(field_texts: dict[str, str]) -> tuple[str, int | None, bool]:
    text = build_labeled_text(field_texts)
    if not text:
        return '', 0, False
    if not TOKEN_AWARE_TRUNCATION:
        return text, None, False

    token_count = count_bge_tokens(text)
    if token_count <= MODEL_MAX_SEQ_LENGTH:
        return text, token_count, False

    truncated_fields = dict(field_texts)
    truncated_fields = shrink_field_to_fit(truncated_fields, 'description', MODEL_MAX_SEQ_LENGTH)
    text = build_labeled_text(truncated_fields)
    token_count = count_bge_tokens(text)
    if token_count <= MODEL_MAX_SEQ_LENGTH:
        return text, token_count, True

    truncated_fields = shrink_field_to_fit(truncated_fields, 'features', MODEL_MAX_SEQ_LENGTH)
    text = build_labeled_text(truncated_fields)
    token_count = count_bge_tokens(text)
    return text, token_count, True


def prepare_metadata_row(row: dict[str, Any], parent_to_final_idx: Mapping[str, int]) -> PreparedTextItem:
    parent_asin = str(row.get('parent_asin', '')).strip()
    field_texts = {field_name: clean_field(field_name, row.get(field_name)) for field_name in TEXT_FIELDS}
    field_texts = deduplicate_fields_across_item(field_texts)
    embedding_text, token_count, was_truncated = fit_text_to_budget(field_texts)
    fail_reason = None if embedding_text else 'empty_cleaned_text'
    return PreparedTextItem(
        parent_asin=parent_asin,
        final_item_idx=int(parent_to_final_idx[parent_asin]),
        store=clean_scalar(row.get('store')),
        price=clean_scalar(row.get('price')),
        embedding_text=embedding_text,
        has_title=bool(field_texts['title']),
        has_categories=bool(field_texts['categories']),
        has_features=bool(field_texts['features']),
        has_description=bool(field_texts['description']),
        embedding_text_chars=len(embedding_text),
        embedding_text_tokens=token_count,
        text_was_token_truncated=was_truncated,
        text_fail_reason=fail_reason,
    )


def item_quality_score(item: PreparedTextItem) -> tuple[int, int, int, int, int]:
    return (int(item.has_title), int(item.has_categories), int(item.has_features), int(item.has_description), item.embedding_text_chars)


def preprocessing_manifest_row(item: PreparedTextItem, include_preview: bool = False) -> dict[str, Any]:
    row = {
        'parent_asin': item.parent_asin,
        'final_item_idx': item.final_item_idx,
        'store': item.store,
        'price': item.price,
        'has_title': int(item.has_title),
        'has_categories': int(item.has_categories),
        'has_features': int(item.has_features),
        'has_description': int(item.has_description),
        'embedding_text_chars': item.embedding_text_chars,
        'embedding_text_tokens': item.embedding_text_tokens,
        'text_was_token_truncated': int(item.text_was_token_truncated),
        'text_fail_reason': item.text_fail_reason,
    }
    if include_preview:
        row['embedding_text_preview'] = item.embedding_text[:1000]
    return row


def write_csv_rows(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0].keys()) if rows else []
    with path.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_filtered_metadata(item_df: pd.DataFrame) -> list[PreparedTextItem]:
    parent_to_final_idx = {str(row.parent_asin): int(row.final_item_idx) for row in item_df.itertuples(index=False)}
    final_parent_set = set(parent_to_final_idx)
    selected: dict[str, PreparedTextItem] = {}
    scanned_rows = 0
    matched_rows = 0
    bad_json_rows = 0

    with META_PATH.open('r', encoding='utf-8') as file:
        progress = tqdm(file, desc='stream metadata', unit='row')
        for line in progress:
            scanned_rows += 1
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                bad_json_rows += 1
                continue

            parent_asin = str(row.get('parent_asin', '')).strip()
            if parent_asin not in final_parent_set:
                continue

            matched_rows += 1
            item = prepare_metadata_row(row, parent_to_final_idx)
            existing = selected.get(parent_asin)
            if existing is None or item_quality_score(item) > item_quality_score(existing):
                selected[parent_asin] = item
            if matched_rows % 10_000 == 0:
                progress.set_postfix(matched=f'{matched_rows:,}')

    missing_parents = final_parent_set - set(selected)
    for parent_asin in missing_parents:
        selected[parent_asin] = PreparedTextItem(
            parent_asin=parent_asin,
            final_item_idx=parent_to_final_idx[parent_asin],
            store=None,
            price=None,
            embedding_text='',
            has_title=False,
            has_categories=False,
            has_features=False,
            has_description=False,
            embedding_text_chars=0,
            embedding_text_tokens=0,
            text_was_token_truncated=False,
            text_fail_reason='missing_metadata',
        )

    items = sorted(selected.values(), key=lambda item: item.final_item_idx)
    write_csv_rows(PREPROCESSING_MANIFEST_PATH, [preprocessing_manifest_row(item) for item in items])
    write_csv_rows(SUMMARY_DIR / 'preprocessed_text_samples.csv', [preprocessing_manifest_row(item, include_preview=True) for item in items[:TEXT_SAMPLE_LIMIT]])

    summary = {
        'stage': 'metadata_preprocessing',
        'metadata_path': str(META_PATH),
        'item_universe_rows': len(item_df),
        'metadata_rows_scanned': scanned_rows,
        'metadata_rows_matched': matched_rows,
        'metadata_bad_json_rows': bad_json_rows,
        'prepared_items': len(items),
        'missing_metadata_items': len(missing_parents),
        'items_with_nonempty_embedding_text': sum(1 for item in items if item.embedding_text),
        'items_with_title': sum(1 for item in items if item.has_title),
        'items_with_categories': sum(1 for item in items if item.has_categories),
        'items_with_features': sum(1 for item in items if item.has_features),
        'items_with_description': sum(1 for item in items if item.has_description),
        'token_aware_truncation': TOKEN_AWARE_TRUNCATION,
        'service_patterns': SERVICE_PATTERNS,
        'created_at_unix': time.time(),
    }
    (SUMMARY_DIR / 'metadata_preprocessing_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps({key: value for key, value in summary.items() if key != 'service_patterns'}, indent=2))
    return items


## Stage B: Resumable BGE Embedding

Chunks are written atomically. Completed chunks are reused on resume. Existing completed `parent_asin` values are skipped, so the run is robust even if the source order changes.

The final `text_manifest.csv` is written after embedding and includes post-embedding status columns. The preprocessing-only manifest is written separately as `text_preprocessing_manifest.csv`.

In [ ]:
def output_schema() -> pa.Schema:
    fields = [
        pa.field('parent_asin', pa.string()),
        pa.field('final_item_idx', pa.int64()),
        pa.field('store', pa.string()),
        pa.field('price', pa.string()),
        pa.field('text_embedding', pa.list_(pa.float32())),
        pa.field('text_embedding_ok', pa.bool_()),
        pa.field('text_fail_reason', pa.string()),
        pa.field('embedding_model', pa.string()),
        pa.field('embedding_dim', pa.int32()),
        pa.field('embedding_dtype', pa.string()),
        pa.field('preprocessing_version', pa.string()),
        pa.field('has_title', pa.bool_()),
        pa.field('has_categories', pa.bool_()),
        pa.field('has_features', pa.bool_()),
        pa.field('has_description', pa.bool_()),
        pa.field('embedding_text_chars', pa.int32()),
        pa.field('embedding_text_tokens', pa.int32()),
        pa.field('text_was_token_truncated', pa.bool_()),
    ]
    if SAVE_EMBEDDING_TEXT_IN_PARQUET:
        fields.append(pa.field('embedding_text', pa.string()))
    return pa.schema(fields)


def chunk_path(chunk_index: int) -> Path:
    return CHUNKS_DIR / f'text_embeddings_Beauty_and_Personal_Care_{chunk_index:05d}.parquet'


def assert_chunk_schema_current(path: Path) -> None:
    schema_names = set(pq.read_schema(path).names)
    required = {'parent_asin', 'final_item_idx', 'text_embedding', 'text_embedding_ok'}
    if not required.issubset(schema_names):
        missing = sorted(required - schema_names)
        raise ValueError(
            f'Existing chunk {path} uses an older/incompatible schema. '
            f'Missing columns: {missing}. Move/delete the old output directory or use a new OUTPUT_DIR.'
        )


def list_chunk_files() -> list[Path]:
    files = sorted(path for path in CHUNKS_DIR.glob('text_embeddings_Beauty_and_Personal_Care_*.parquet') if not path.name.endswith('.tmp.parquet'))
    for path in files:
        assert_chunk_schema_current(path)
    return files


def chunk_index_from_path(path: Path) -> int:
    try:
        return int(path.stem.rsplit('_', 1)[-1])
    except ValueError:
        return -1


def existing_output_state() -> tuple[set[str], int, int]:
    embedded_parents: set[str] = set()
    files = list_chunk_files()
    for path in files:
        table = pq.read_table(path, columns=['parent_asin'])
        embedded_parents.update(str(value) for value in table.column('parent_asin').to_pylist())
    next_chunk_index = max([chunk_index_from_path(path) for path in files], default=-1) + 1
    return embedded_parents, next_chunk_index, len(files)


def zero_embedding() -> np.ndarray:
    return np.zeros((EMBEDDING_DIM,), dtype=np.float32)


def record_from_item(item: PreparedTextItem, embedding: np.ndarray, ok: bool, fail_reason: str | None) -> dict[str, Any]:
    record = {
        'parent_asin': item.parent_asin,
        'final_item_idx': int(item.final_item_idx),
        'store': item.store,
        'price': item.price,
        'text_embedding': embedding.astype(np.float32).tolist(),
        'text_embedding_ok': bool(ok),
        'text_fail_reason': fail_reason,
        'embedding_model': MODEL_NAME,
        'embedding_dim': EMBEDDING_DIM,
        'embedding_dtype': EMBEDDING_DTYPE,
        'preprocessing_version': PREPROCESSING_VERSION,
        'has_title': item.has_title,
        'has_categories': item.has_categories,
        'has_features': item.has_features,
        'has_description': item.has_description,
        'embedding_text_chars': int(item.embedding_text_chars),
        'embedding_text_tokens': int(item.embedding_text_tokens or 0),
        'text_was_token_truncated': item.text_was_token_truncated,
    }
    if SAVE_EMBEDDING_TEXT_IN_PARQUET:
        record['embedding_text'] = item.embedding_text
    return record


def encode_items_safely(items: list[PreparedTextItem]) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    text_positions = [(index, item) for index, item in enumerate(items) if item.embedding_text]
    encoded_by_index: dict[int, tuple[np.ndarray, bool, str | None]] = {}

    if text_positions:
        texts = [item.embedding_text for _, item in text_positions]
        try:
            embeddings = model.encode(texts, batch_size=BATCH_SIZE, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
            for (index, _), embedding in zip(text_positions, embeddings):
                encoded_by_index[index] = (embedding, True, None)
        except Exception as batch_error:
            print(f'Batch encode failed; retrying per item. Error: {batch_error}')
            for index, item in text_positions:
                try:
                    embedding = model.encode([item.embedding_text], batch_size=1, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0].astype(np.float32)
                    encoded_by_index[index] = (embedding, True, None)
                except Exception as item_error:
                    encoded_by_index[index] = (zero_embedding(), False, f'encoder_error: {item_error}')

    for index, item in enumerate(items):
        if item.embedding_text:
            embedding, ok, fail_reason = encoded_by_index.get(index, (zero_embedding(), False, 'encoder_error_unknown'))
            records.append(record_from_item(item, embedding, ok, fail_reason))
        else:
            records.append(record_from_item(item, zero_embedding(), False, item.text_fail_reason or 'empty_cleaned_text'))
    return records


def atomic_write_parquet(records: list[dict[str, Any]], output_path: Path) -> None:
    temp_path = output_path.with_name(output_path.name + '.tmp.parquet')
    if temp_path.exists():
        temp_path.unlink()
    table = pa.Table.from_pylist(records, schema=output_schema())
    pq.write_table(table, temp_path, compression='zstd')
    os.replace(temp_path, output_path)


def count_embedding_status_from_chunks() -> tuple[int, int, float | None]:
    ok_count = 0
    failed_count = 0
    for path in list_chunk_files():
        table = pq.read_table(path, columns=['text_embedding_ok'])
        values = table.column('text_embedding_ok').to_pylist()
        ok_count += sum(1 for value in values if value)
        failed_count += sum(1 for value in values if not value)
    total = ok_count + failed_count
    ok_rate = ok_count / total if total else None
    return ok_count, failed_count, ok_rate


def write_embedding_manifest_from_chunks(prepared_items: list[PreparedTextItem]) -> Path:
    prepared_by_parent = {item.parent_asin: item for item in prepared_items}
    rows: list[dict[str, Any]] = []
    manifest_columns = [
        'parent_asin',
        'final_item_idx',
        'store',
        'price',
        'text_embedding_ok',
        'text_fail_reason',
        'embedding_model',
        'embedding_dim',
        'embedding_dtype',
        'preprocessing_version',
        'has_title',
        'has_categories',
        'has_features',
        'has_description',
        'embedding_text_chars',
        'embedding_text_tokens',
        'text_was_token_truncated',
    ]
    for path in list_chunk_files():
        table = pq.read_table(path, columns=manifest_columns)
        for row in table.to_pylist():
            item = prepared_by_parent.get(str(row['parent_asin']))
            row['embedding_text_preview'] = item.embedding_text[:300] if item is not None else ''
            rows.append(row)
    rows.sort(key=lambda row: (row.get('final_item_idx') or 10**18, row['parent_asin']))
    write_csv_rows(TEXT_MANIFEST_PATH, rows)
    return TEXT_MANIFEST_PATH


def write_embedding_run_summary(summary: dict[str, Any]) -> None:
    path = SUMMARY_DIR / 'text_embedding_run_summary.json'
    path.write_text(json.dumps(summary, indent=2), encoding='utf-8')


def run_embedding(prepared_items: list[PreparedTextItem]) -> dict[str, Any]:
    existing_parents, next_chunk_index, existing_files = existing_output_state()
    items_to_embed = [item for item in prepared_items if item.parent_asin not in existing_parents]

    print(f'Prepared items: {len(prepared_items):,}')
    print(f'Existing chunks: {existing_files:,}')
    print(f'Existing embedded parent_asin rows: {len(existing_parents):,}')
    print(f'Items left to write: {len(items_to_embed):,}')

    output_records: list[dict[str, Any]] = []
    chunk_index = next_chunk_index
    embedded_this_run = 0
    started_at = time.time()

    progress = tqdm(range(0, len(items_to_embed), BATCH_SIZE), desc='embed text batches', unit='batch')
    for start in progress:
        batch_items = items_to_embed[start:start + BATCH_SIZE]
        output_records.extend(encode_items_safely(batch_items))
        embedded_this_run += len(batch_items)

        while len(output_records) >= CHUNK_SIZE:
            chunk_records = output_records[:CHUNK_SIZE]
            output_records = output_records[CHUNK_SIZE:]
            path = chunk_path(chunk_index)
            if path.exists():
                raise FileExistsError(f'Refusing to overwrite existing chunk: {path}')
            atomic_write_parquet(chunk_records, path)
            chunk_index += 1
            progress.set_postfix(written=f'{embedded_this_run:,}')
            gc.collect()

    if output_records:
        path = chunk_path(chunk_index)
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite existing chunk: {path}')
        atomic_write_parquet(output_records, path)
        chunk_index += 1

    final_files = list_chunk_files()
    final_rows = sum(pq.ParquetFile(path).metadata.num_rows for path in final_files)
    ok_count, failed_count, ok_rate = count_embedding_status_from_chunks()
    manifest_path = write_embedding_manifest_from_chunks(prepared_items)
    summary = {
        'stage': 'embedding',
        'run_mode': RUN_MODE,
        'output_dir': str(OUTPUT_DIR),
        'chunks_dir': str(CHUNKS_DIR),
        'text_manifest_path': str(manifest_path),
        'model_name': MODEL_NAME,
        'model_max_seq_length': MODEL_MAX_SEQ_LENGTH,
        'embedding_dim': EMBEDDING_DIM,
        'embedding_dtype': EMBEDDING_DTYPE,
        'normalized_embeddings': True,
        'preprocessing_version': PREPROCESSING_VERSION,
        'batch_size': BATCH_SIZE,
        'chunk_size': CHUNK_SIZE,
        'max_final_items': MAX_FINAL_ITEMS,
        'prepared_items': len(prepared_items),
        'existing_rows_before_run': len(existing_parents),
        'embedded_this_run': embedded_this_run,
        'final_chunk_files': len(final_files),
        'final_output_rows': final_rows,
        'text_embedding_ok_rows': ok_count,
        'text_embedding_failed_or_zero_rows': failed_count,
        'text_embedding_ok_rate': ok_rate,
        'elapsed_minutes': round((time.time() - started_at) / 60, 3),
        'updated_at_unix': time.time(),
    }
    write_embedding_run_summary(summary)
    print(json.dumps(summary, indent=2))
    return summary


## Stage C: Combined Parquet, QA, and Tensor Helpers

The chunked Parquet files are the canonical resumable cache. The combined Parquet file is useful for downstream reads.

The debug tensor is indexed by dense `final_item_idx` over the final item universe. The production tensor should be aligned to RecBole's internal item IDs by calling `build_recbole_aligned_tensor_from_dataset(dataset, item_df)` after the RecBole dataset is loaded.

In [ ]:
def write_combined_parquet() -> Path | None:
    files = list_chunk_files()
    if not files:
        print('No chunk files found; skipping combined parquet write.')
        return None

    temp_path = COMBINED_PARQUET_PATH.with_name(COMBINED_PARQUET_PATH.name + '.tmp')
    if temp_path.exists():
        temp_path.unlink()

    writer: pq.ParquetWriter | None = None
    try:
        for path in tqdm(files, desc='combine chunks', unit='file'):
            table = pq.read_table(path)
            if writer is None:
                writer = pq.ParquetWriter(temp_path, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    os.replace(temp_path, COMBINED_PARQUET_PATH)
    print(f'Wrote combined parquet: {COMBINED_PARQUET_PATH}')
    return COMBINED_PARQUET_PATH


def count_chunk_rows() -> int:
    return sum(pq.ParquetFile(path).metadata.num_rows for path in list_chunk_files())


def run_output_qa(prepared_items: list[PreparedTextItem]) -> dict[str, Any]:
    files = list_chunk_files()
    actual_rows = count_chunk_rows()
    expected_rows = len(prepared_items)
    difference = actual_rows - expected_rows
    expected_parents = {item.parent_asin for item in prepared_items}

    schema_text = ''
    vector_type_is_float32 = False
    vector_length_bad_rows = 0
    embedding_dim_bad_rows = 0
    non_finite_rows = 0
    failed_nonzero_rows = 0
    duplicate_parent_rows = 0
    unexpected_parent_rows = 0
    missing_parent_rows = 0
    ok_count = 0
    failed_count = 0
    norms: list[float] = []
    seen_parents: set[str] = set()

    if files:
        first_schema = pq.read_schema(files[0])
        schema_text = str(first_schema)
        try:
            vector_type_is_float32 = first_schema.field('text_embedding').type.value_type == pa.float32()
        except Exception:
            vector_type_is_float32 = False

        for path in tqdm(files, desc='QA chunks', unit='file'):
            table = pq.read_table(path, columns=['parent_asin', 'text_embedding', 'text_embedding_ok', 'embedding_dim'])
            for row in table.to_pylist():
                parent_asin = str(row['parent_asin'])
                if parent_asin in seen_parents:
                    duplicate_parent_rows += 1
                seen_parents.add(parent_asin)
                if parent_asin not in expected_parents:
                    unexpected_parent_rows += 1

                vector = np.asarray(row['text_embedding'], dtype=np.float32)
                if vector.shape != (EMBEDDING_DIM,):
                    vector_length_bad_rows += 1
                if int(row['embedding_dim']) != EMBEDDING_DIM:
                    embedding_dim_bad_rows += 1
                if not np.isfinite(vector).all():
                    non_finite_rows += 1

                norm = float(np.linalg.norm(vector))
                if row['text_embedding_ok']:
                    ok_count += 1
                    norms.append(norm)
                else:
                    failed_count += 1
                    if norm != 0.0:
                        failed_nonzero_rows += 1

    missing_parent_rows = len(expected_parents - seen_parents)
    norm_min = min(norms) if norms else None
    norm_max = max(norms) if norms else None
    ok_rate = ok_count / (ok_count + failed_count) if (ok_count + failed_count) else None
    norms_close_to_one = bool(norms) and norm_min is not None and norm_max is not None and norm_min >= 0.98 and norm_max <= 1.02

    checks = {
        'row_count_matches_expected': difference == 0,
        'vector_type_is_float32': vector_type_is_float32,
        'all_vector_lengths_match_embedding_dim': vector_length_bad_rows == 0,
        'all_embedding_dim_values_match': embedding_dim_bad_rows == 0,
        'all_vectors_finite': non_finite_rows == 0,
        'failed_rows_are_zero_vectors': failed_nonzero_rows == 0,
        'no_duplicate_parent_asin_rows': duplicate_parent_rows == 0,
        'no_unexpected_parent_asin_rows': unexpected_parent_rows == 0,
        'no_missing_parent_asin_rows': missing_parent_rows == 0,
        'successful_vector_norms_close_to_one': norms_close_to_one,
    }
    status = 'PASS' if all(checks.values()) else 'FAIL'

    qa = {
        'expected_prepared_items': expected_rows,
        'actual_vector_rows': actual_rows,
        'difference_actual_minus_expected': difference,
        'status': status,
        'checks': checks,
        'chunk_files': len(files),
        'text_embedding_ok_rows': ok_count,
        'text_embedding_failed_or_zero_rows': failed_count,
        'text_embedding_ok_rate': ok_rate,
        'successful_vector_norm_min': norm_min,
        'successful_vector_norm_max': norm_max,
        'vector_length_bad_rows': vector_length_bad_rows,
        'embedding_dim_bad_rows': embedding_dim_bad_rows,
        'non_finite_rows': non_finite_rows,
        'failed_nonzero_rows': failed_nonzero_rows,
        'duplicate_parent_rows': duplicate_parent_rows,
        'unexpected_parent_rows': unexpected_parent_rows,
        'missing_parent_rows': missing_parent_rows,
        'schema': schema_text,
        'updated_at_unix': time.time(),
    }

    (SUMMARY_DIR / 'text_vector_row_comparison.json').write_text(json.dumps(qa, indent=2), encoding='utf-8')
    flat_qa = {key: value for key, value in qa.items() if key not in {'schema', 'checks'}}
    flat_qa.update({f'check_{key}': value for key, value in checks.items()})
    write_csv_rows(SUMMARY_DIR / 'text_vector_row_comparison.csv', [flat_qa])
    print(json.dumps(flat_qa, indent=2))
    return qa


def load_embedding_records_from_chunks() -> dict[str, dict[str, Any]]:
    lookup: dict[str, dict[str, Any]] = {}
    columns = ['parent_asin', 'text_embedding', 'text_embedding_ok', 'text_fail_reason']
    for path in tqdm(list_chunk_files(), desc='load chunk embeddings', unit='file'):
        table = pq.read_table(path, columns=columns)
        for row in table.to_pylist():
            lookup[str(row['parent_asin'])] = {
                'vector': np.asarray(row['text_embedding'], dtype=np.float32),
                'text_embedding_ok': bool(row['text_embedding_ok']),
                'text_fail_reason': row['text_fail_reason'],
            }
    return lookup


def build_final_item_idx_debug_tensor(item_df: pd.DataFrame, output_path: Path = FINAL_INDEX_DEBUG_TENSOR_PATH) -> Path:
    embedding_lookup = load_embedding_records_from_chunks()
    max_final_item_idx = int(item_df['final_item_idx'].max())
    tensor = torch.zeros((max_final_item_idx + 1, EMBEDDING_DIM), dtype=torch.float32)

    for row in tqdm(item_df.itertuples(index=False), total=len(item_df), desc='build final_item_idx tensor'):
        parent_asin = str(row.parent_asin)
        final_item_idx = int(row.final_item_idx)
        record = embedding_lookup.get(parent_asin)
        if record is not None:
            tensor[final_item_idx] = torch.from_numpy(record['vector'].astype(np.float32))

    torch.save(tensor, output_path)
    print(f'Wrote debug tensor: {output_path}, shape={tuple(tensor.shape)}, dtype={tensor.dtype}')
    return output_path


def build_recbole_aligned_tensor_from_token_map(
    token_to_internal_id: Mapping[Any, int],
    item_df: pd.DataFrame,
    output_path: Path = RECBOLE_ALIGNED_TENSOR_PATH,
) -> Path:
    embedding_lookup = load_embedding_records_from_chunks()
    final_idx_to_parent = {str(int(row.final_item_idx)): str(row.parent_asin) for row in item_df.itertuples(index=False)}
    parent_to_final_idx = {str(row.parent_asin): int(row.final_item_idx) for row in item_df.itertuples(index=False)}
    n_items = max(int(internal_id) for internal_id in token_to_internal_id.values()) + 1
    tensor = torch.zeros((n_items, EMBEDDING_DIM), dtype=torch.float32)
    map_rows: list[dict[str, Any]] = []

    for token, internal_id_raw in token_to_internal_id.items():
        internal_id = int(internal_id_raw)
        token_str = str(token)
        if internal_id == 0 or token_str in {'[PAD]', 'PAD'}:
            continue

        if token_str in embedding_lookup:
            parent_asin = token_str
            final_item_idx = parent_to_final_idx.get(parent_asin)
        else:
            parent_asin = final_idx_to_parent.get(token_str)
            final_item_idx = int(token_str) if token_str.isdigit() and token_str in final_idx_to_parent else None

        text_embedding_ok = False
        text_fail_reason = 'missing_embedding_row'
        has_text = False
        if parent_asin is not None and parent_asin in embedding_lookup:
            record = embedding_lookup[parent_asin]
            tensor[internal_id] = torch.from_numpy(record['vector'].astype(np.float32))
            text_embedding_ok = bool(record['text_embedding_ok'])
            text_fail_reason = record['text_fail_reason']
            has_text = bool(torch.linalg.norm(tensor[internal_id]).item() > 0)

        map_rows.append({
            'recbole_internal_item_id': internal_id,
            'item_id_token': token_str,
            'final_item_idx': final_item_idx,
            'parent_asin': parent_asin,
            'has_text': int(has_text),
            'text_embedding_ok': int(text_embedding_ok),
            'text_fail_reason': text_fail_reason,
        })

    torch.save(tensor, output_path)
    write_csv_rows(OUTPUT_DIR / 'beauty_item_id_parent_asin_map.csv', map_rows)
    print(f'Wrote RecBole-aligned tensor: {output_path}, shape={tuple(tensor.shape)}, dtype={tensor.dtype}')
    return output_path


def build_recbole_aligned_tensor_from_dataset(
    dataset: Any,
    item_df: pd.DataFrame,
    output_path: Path = RECBOLE_ALIGNED_TENSOR_PATH,
) -> Path:
    item_field = getattr(dataset, 'iid_field', 'item_id')
    token_to_internal_id = dataset.field2token_id[item_field]
    return build_recbole_aligned_tensor_from_token_map(token_to_internal_id, item_df, output_path)


def run_tensor_qa(tensor_path: Path, expected_dim: int = EMBEDDING_DIM) -> dict[str, Any]:
    tensor = torch.load(tensor_path, map_location='cpu')
    checks = {
        'tensor_is_float32': tensor.dtype == torch.float32,
        'tensor_is_2d': tensor.ndim == 2,
        'tensor_embedding_dim_matches': tensor.ndim == 2 and int(tensor.shape[1]) == expected_dim,
        'row_0_all_zero': bool(torch.all(tensor[0] == 0).item()) if tensor.shape[0] > 0 else False,
        'tensor_has_no_nan': bool(not torch.isnan(tensor).any().item()),
        'tensor_has_no_inf': bool(not torch.isinf(tensor).any().item()),
    }
    qa = {
        'tensor_path': str(tensor_path),
        'shape': list(tensor.shape),
        'dtype': str(tensor.dtype),
        'status': 'PASS' if all(checks.values()) else 'FAIL',
        'checks': checks,
        'updated_at_unix': time.time(),
    }
    qa_path = SUMMARY_DIR / f'{tensor_path.stem}_qa.json'
    qa_path.write_text(json.dumps(qa, indent=2), encoding='utf-8')
    print(json.dumps(qa, indent=2))
    return qa


def write_consolidated_summary(
    item_df: pd.DataFrame,
    prepared_items: list[PreparedTextItem],
    embedding_summary: dict[str, Any],
    qa_summary: dict[str, Any],
    combined_parquet_path: Path | None,
    debug_tensor_path: Path | None,
    recbole_tensor_path: Path | None,
) -> Path:
    payload = {
        'run_mode': RUN_MODE,
        'created_at_unix': time.time(),
        'project_dir': str(PROJECT_DIR),
        'review_path': str(REVIEW_PATH),
        'metadata_path': str(META_PATH),
        'output_dir': str(OUTPUT_DIR),
        'model_name': MODEL_NAME,
        'model_max_seq_length': MODEL_MAX_SEQ_LENGTH,
        'embedding_dim': EMBEDDING_DIM,
        'embedding_dtype': EMBEDDING_DTYPE,
        'preprocessing_version': PREPROCESSING_VERSION,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'train_end_cutoff_date': TRAIN_END_CUTOFF_DATE,
        'valid_end_cutoff_date': VALID_END_CUTOFF_DATE,
        'user_min_reviews': USER_MIN_REVIEWS,
        'warm_user_min_reviews': WARM_USER_MIN_REVIEWS,
        'warm_item_min_reviews': WARM_ITEM_MIN_REVIEWS,
        'max_review_rows': MAX_REVIEW_ROWS,
        'max_final_items': MAX_FINAL_ITEMS,
        'full_item_universe_rows_available': int(len(item_universe_df)),
        'items_selected_this_run': int(len(item_df)),
        'prepared_items': len(prepared_items),
        'items_with_nonempty_embedding_text': sum(1 for item in prepared_items if item.embedding_text),
        'embedding_summary': embedding_summary,
        'qa_summary': {key: value for key, value in qa_summary.items() if key != 'schema'},
        'outputs': {
            'item_universe_csv': str(ITEM_UNIVERSE_PATH),
            'preprocessing_manifest_csv': str(PREPROCESSING_MANIFEST_PATH),
            'text_manifest_csv': str(TEXT_MANIFEST_PATH),
            'chunks_dir': str(CHUNKS_DIR),
            'combined_parquet': str(combined_parquet_path) if combined_parquet_path is not None else None,
            'final_item_idx_debug_tensor': str(debug_tensor_path) if debug_tensor_path is not None else None,
            'recbole_aligned_tensor': str(recbole_tensor_path) if recbole_tensor_path is not None else None,
            'summary_json': str(SUMMARY_JSON_PATH),
        },
        'recbole_tensor_status': 'created' if recbole_tensor_path is not None else 'pending_dataset_context',
    }
    SUMMARY_JSON_PATH.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(f'Wrote consolidated summary: {SUMMARY_JSON_PATH}')
    return SUMMARY_JSON_PATH


## Run Pipeline

For a first validation run, keep `MAX_FINAL_ITEMS = 500` in the config. This still derives the full item universe, but embeds only 500 items. For the full production embedding run, set:

```python
MAX_FINAL_ITEMS = None
```

Then rerun all cells. Existing completed chunks will be reused.

If a RecBole `dataset` variable is already loaded in this notebook, the final cell will also build the RecBole-aligned tensor and item-id map. Otherwise, those outputs are intentionally marked as pending in `summary.json`.

In [ ]:
full_item_universe_df = ensure_item_universe()
if MAX_FINAL_ITEMS is not None:
    run_item_universe_df = full_item_universe_df.head(MAX_FINAL_ITEMS).copy()
else:
    run_item_universe_df = full_item_universe_df.copy()

print(f'Items selected for this run: {len(run_item_universe_df):,}')
prepared_items = load_filtered_metadata(run_item_universe_df)
embedding_summary = run_embedding(prepared_items)

combined_parquet_path = write_combined_parquet() if WRITE_COMBINED_PARQUET else None
qa_summary = run_output_qa(prepared_items)

debug_tensor_path: Path | None = None
if BUILD_FINAL_INDEX_DEBUG_TENSOR:
    debug_tensor_path = build_final_item_idx_debug_tensor(run_item_universe_df)
    run_tensor_qa(debug_tensor_path)

recbole_tensor_path: Path | None = None
if AUTO_BUILD_RECBOLE_TENSOR_IF_DATASET_EXISTS and 'dataset' in globals():
    recbole_tensor_path = build_recbole_aligned_tensor_from_dataset(dataset, run_item_universe_df)
    run_tensor_qa(recbole_tensor_path)
else:
    print('RecBole dataset object not found; RecBole-aligned tensor is pending dataset context.')

write_consolidated_summary(
    item_df=run_item_universe_df,
    prepared_items=prepared_items,
    embedding_summary=embedding_summary,
    qa_summary=qa_summary,
    combined_parquet_path=combined_parquet_path,
    debug_tensor_path=debug_tensor_path,
    recbole_tensor_path=recbole_tensor_path,
)

print('Done.')
